# **Lab project 2: Advanced deep generative models**

**Pablo Martínez Olmos, Vanessa Gómez Verdejo,  Alejandro Lancho Serrano, Emilio Parrado Hernández**

Departamento de Teoría de la Señal y Comunicaciones

**Universidad Carlos III de Madrid**

<img src='https://upload.wikimedia.org/wikipedia/commons/8/88/Acronimo_nombre3l.jpg' width=300 />


# Part 1: Conditional Generation in Deep Generative Models

In this first part, you will progressively explore the design and training of Variational Autoencoders (VAEs) using the **Fashion MNIST** dataset ([torchvision.datasets.FashionMNIST](https://pytorch.org/vision/stable/generated/torchvision.datasets.FashionMNIST.html)). The first part will guide you through building a standard VAE, experimenting with different likelihoods, and analyzing the latent space. As you advance, you will extend your models to more expressive approaches, culminating in the implementation and analysis of conditional VAEs. Each section will build upon the previous, helping you understand how architectural choices and conditioning impact generative modeling and representation learning.

The notebook is structured so that each section introduces a new concept or modeling challenge, leading you step by step from basic VAEs to conditional generative models.

## Exercise 1: Standard VAE Training and Likelihood Comparison

In this exercise, you will:
- Implement and train a standard VAE on FashionMNIST.
- Compare two likelihoods for the decoder: Gaussian and soft Bernoulli.
- Analyze and discuss the reconstruction quality and latent space for each likelihood.

We import the necessary libraries for implementing Variational Autoencoders (VAEs).

In [10]:
from torchvision.datasets import FashionMNIST
from torchvision import transforms

import torch
import torch.nn as nn

import numpy as np

from torchvision import datasets, transforms
from torch.utils.data import Dataset

import pytorch_lightning as pl

In [ ]:
train_dataset = FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

In [ ]:
np.min(train_dataset[0][0].numpy())

np.float32(0.0)

We download the dataset.

In [5]:
class Fashion_MNIST(Dataset):
    def __init__(self, mode='train'):
        if mode == 'train':
            self.data = FashionMNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())
        else: # Using test split for evaluation
            self.data = FashionMNIST(root="./data", train=False, download=True, transform=transforms.ToTensor())

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img, _ = self.data[idx]
        # Flatten the 28x28 image to a 784-dimensional vector
        return img.view(-1)



train_dataset = Fashion_MNIST(mode='train')
val_dataset = Fashion_MNIST(mode='val')
test_dataset = Fashion_MNIST(mode='test')


In [8]:
len(test_dataset)

10000

VAE code

In [9]:
EPS = 1e-8


class Encoder(nn.Module):
    def __init__(self, encoder_net):
        super(Encoder, self).__init__()

        # The encoder network (e.g., MLP) that outputs concatenated [mu, log_var]
        self.encoder = encoder_net

    def reparameterization(self, mu, log_var):
        # Reparameterization trick:
        # Sample z ~ N(mu, sigma^2) using z = mu + sigma * epsilon,
        # where epsilon ~ N(0, I)
        std = torch.exp(0.5 * log_var)        # Convert log variance to standard deviation
        eps = torch.randn_like(std)           # Sample epsilon with the same shape as std (L=1)
        return mu + std * eps                 # Sample z

    def encode(self, x):
        # Forward input x through the encoder network to get mu and log_var
        h_e = self.encoder(x)                 # Output is of size 2M
        mu_e, log_var_e = torch.chunk(h_e, 2, dim=1)  # Split into two parts: mean and log variance
        return mu_e, log_var_e

    def sample(self, x=None, mu_e=None, log_var_e=None):
        # Return a sample from the approximate posterior
        # Can either compute mu/log_var from x or take them as input
        if (mu_e is None) and (log_var_e is None):
            mu_e, log_var_e = self.encode(x)
        else:
            if (mu_e is None) or (log_var_e is None):
                raise ValueError('mu and log_var can\'t be None!')
        z = self.reparameterization(mu_e, log_var_e)
        return z



class Decoder(nn.Module):
    def __init__(self, decoder_net, fixed_var=1.0):
        super(Decoder, self).__init__()
        self.decoder = decoder_net
        self.var = fixed_var  # Fixed variance

    def decode(self, z):
        # We only learn the mean of p(x|z), variance is fixed
        mu = self.decoder(z)
        return mu

    def sample(self, z):
        mu = self.decode(z)
        std = torch.sqrt(torch.tensor(self.var)).to(mu.device)
        eps = torch.randn_like(mu)
        return mu + eps * std

    def log_prob(self, x, z):
        mu = self.decode(z)
        # log N(x | mu, sigma^2 I) with fixed variance
        log_prob = -0.5 * ((x - mu) ** 2) / self.var - 0.5 * torch.log(2 * torch.pi * torch.tensor(self.var))
        # Note that first term is the MSE and second term is a constant depending on variance
        return log_prob.sum(dim=-1)

class Prior(nn.Module):
    def __init__(self, M):
        super(Prior, self).__init__()
        # M is the dimensionality of the latent space
        self.M = M

    def sample(self, batch_size):
        # Sample from a standard normal distribution (mean=0, std=1)
        z = torch.randn((batch_size, self.M))  # Shape: (batch_size, latent_dim)
        return z

class VAE(nn.Module):
    def __init__(self, encoder_net, decoder_net, M=16):
        super(VAE, self).__init__()

        self.encoder = Encoder(encoder_net=encoder_net)
        self.decoder = Decoder(decoder_net=decoder_net)
        self.prior = Prior(M=M)

    def forward(self, x):
        # generate a latent variable, for each training sample, with the encoder
        mu_e, log_var_e = self.encoder.encode(x)
        z = self.encoder.sample(mu_e=mu_e, log_var_e=log_var_e)

        # ELBO
        RE = self.decoder.log_prob(x, z)
        KL = - 0.5 * torch.sum(torch.exp(log_var_e) + mu_e**2 - 1 - log_var_e, dim=1)
        return -(RE + KL).sum()


    def sample(self, batch_size=64):
        # Generate a latent variable with the prior
        z = self.prior.sample(batch_size=batch_size)
        return self.decoder.sample(z)





Lighting model

In [ ]:
class LitGenericModel(pl.LightningModule):
    def __init__(self, model, loss, score ,lr=1e-3):
        super().__init__()
        self.save_hyperparameters(ignore=['model', 'loss', 'accuracy'])  # logs hyperparameters automatically

        self.model = model
        self.loss = loss
        self.score = score

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch):
        xs, ys = batch
        ys_estimate = self(xs)
        ys_estimate = ys_estimate.squeeze(1)
        loss = self.loss(ys_estimate, ys)
        acc = self.score(ys_estimate, ys)
  
        return loss, acc, ys_estimate, ys

    def training_step(self, batch, batch_idx):
        loss, acc, ys_estimate, ys = self._shared_step(batch)
        self.log('loss.train', loss, on_epoch=True, on_step=False, prog_bar=False)
        self.log('score.train', acc, on_epoch=True, on_step=False, prog_bar=False)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, acc, ys_estimate, ys = self._shared_step(batch)
        self.log('loss.valid', loss, on_epoch=True, on_step=False, prog_bar=False)
        self.log('score.valid', acc, on_epoch=True, on_step=False, prog_bar=False)
        return loss

    def test_step(self, batch, batch_idx):
        loss, acc, ys_estimate, ys = self._shared_step(batch)
        self.log('loss.test', loss, on_epoch=True, on_step=False, prog_bar=False)
        self.log('score.test', acc, on_epoch=True, on_step=False, prog_bar=False)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

## Exercise 2: Advanced VAE - GMM-VAE or VQ-VAE

Based on your results from Exercise 1, choose the best encoder likelihood and complete the following tasks:
1. Implement and train either a GMM-VAE or a VQ-VAE using the best likelihood from Exercise 1.
2. Visualize and analyze the latent space of your chosen model.
3. Compare the results with the standard VAE.

## Exercise 3: Comparison with Conditional Models

In this section, you will:
- Compare the latent spaces of your advanced VAE (GMM-VAE or VQ-VAE) with a conditional VAE.
- Discuss similarities and differences in the latent representations.




### Conditional Generative Models
There are three types of variables in a deep conditional generative model (CGM): input variables $\mathbf{y}$, output variables $\mathbf{x}$, and latent variables $\mathbf{z}$. The conditional generative process of the model is as follows: for a given observation $\mathbf{y}$, $\mathbf{z}$ is drawn from the prior distribution $p_\theta(\mathbf{z}|\mathbf{y})$, and the output $\mathbf{x}$ is generated from the distribution $p_\theta(\mathbf{x}|\mathbf{y}, \mathbf{z})$. The prior of the latent variables $\mathbf{z}$ is modulated by the input $\mathbf{y}$ in our formulation; however, the constraint can be easily relaxed to make the latent variables statistically independent of input variables, i.e., $p_\theta(\mathbf{z}|\mathbf{y}) = p_\theta(\mathbf{z})$.


In [11]:
from graphviz import Digraph
from IPython.display import display, Markdown


def is_google_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


# Run this if we are in google colab
if is_google_colab():
    # First diagram: Generative model
    g1 = Digraph('Generative_Model')
    g1.attr(rankdir='LR')
    g1.node('Y')
    g1.node('X')
    g1.node('Z')
    g1.edge('Y', 'Z', label='p_θ(z|y)')
    g1.edge('Z', 'X')
    g1.edge('Y', 'X', label='p_θ(x|y,z)')

    # Second diagram: Recognition (inference) model
    g2 = Digraph('Recognition_Model')
    g2.attr(rankdir='LR')
    g2.node('Y')
    g2.node('X')
    g2.node('Z')
    g2.edge('Y', 'Z')
    g2.edge('X', 'Z', label='q_ϕ(z|y,x)')

    # Display with titles
    display(Markdown("### Conditional Generative Model"))
    display(g1)
    display(Markdown("### Recognition Model"))
    display(g2)


ModuleNotFoundError: No module named 'graphviz'

Deep CGMs are trained to maximize the conditional log-likelihood. Often the objective function is intractable, and we apply the stochastic variational inference framework to train the model. The variational lower bound of the model is written as follows:

\begin{equation}
\log p_\theta(\mathbf{x}|\mathbf{y}) \geq -\mathrm{KL}\left(q_\phi(\mathbf{z}|\mathbf{x}, \mathbf{y}) \| p_\theta(\mathbf{z}|\mathbf{y})\right) + \mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x}, \mathbf{y})}\left[\log p_\theta(\mathbf{x}|\mathbf{y}, \mathbf{z})\right]
\end{equation}

and the empirical lower bound is written as:

\begin{equation}
\widetilde{\mathcal{L}}_{\text{CVAE}}(\mathbf{x}, \mathbf{y}; \theta, \phi) = -\mathrm{KL}\left(q_\phi(\mathbf{z}|\mathbf{x}, \mathbf{y}) \| p_\theta(\mathbf{z}|\mathbf{y})\right) + \frac{1}{L} \sum_{l=1}^{L} \log p_\theta(\mathbf{x}|\mathbf{y}, \mathbf{z}^{(l)}),
\end{equation}

where $\mathbf{z}^{(l)} = g_\phi(\mathbf{x}, \mathbf{y}; \epsilon^{(l)})$, $\epsilon^{(l)} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and $L$ is the number of samples. We call this model *conditional variational auto-encoder* (CVAE). The CVAE is composed of multiple MLPs, such as recognition network $q_\phi(\mathbf{z}|\mathbf{x}, \mathbf{y})$, (conditional) prior network $p_\theta(\mathbf{z}|\mathbf{y})$, and generation network $p_\theta(\mathbf{x}|\mathbf{y}, \mathbf{z})$.


### Implementing a Conditional VAE

#### **1. Implement a CVAE with Independent Prior**

- Use a **standard Gaussian prior**: $p(\mathbf{z}) = \mathcal{N}(0, I)$
- In this setting:
  - $\mathbf{y}$: class label (0–9)
  - $\mathbf{x}$: FMNIST image
- Feed the label $\mathbf{y}$ to the **decoder** $p_\theta(\mathbf{x} \mid \mathbf{y}, \mathbf{z})$, using either:
  - One-hot encoding
  - A learnable [embedding layer](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)



#### **2. Modify the Prior to be Label-Dependent**

- Replace the independent prior with a **label-conditional prior**:

$$
p_\theta(\mathbf{z} \mid \mathbf{y})
$$

- Implement this using a small MLP that takes the class label as input (one-hot or embedding layer) and outputs $\mu(\mathbf{y})$, $\log\sigma^2(\mathbf{y})$
- Update the KL divergence accordingly:

$$
\mathrm{KL}\left(q_\phi(\mathbf{z} \mid \mathbf{x}, \mathbf{y}) \,\|\, p_\theta(\mathbf{z} \mid \mathbf{y})\right)
$$

In both cases:
- Plot the evolution of:
  - Total loss
  - KL divergence
  - Reconstruction loss
- After training:
  - Generate samples for each class label $\mathbf{y}$
  - Use the recognition network $q_\phi(\mathbf{z} \mid \mathbf{x}, \mathbf{y})$ to encode real test images
  - Visualize the resulting latent space using **t-SNE**, colored by class



## **Part 2: Missing Data Imputation with VAEs**

In this exercise, you'll work with the [Spambase dataset](https://archive.ics.uci.edu/dataset/94/spambase) from the UCI Machine Learning Repository. The goal is to use **Variational Autoencoders (VAEs)** to impute artificially introduced missing values in tabular data.

---

### **Step-by-Step Instructions**

1. **Download and Prepare the Dataset**
   - Load the Spambase dataset.
   - Normalize all features (ignore the `spam` label for now).
   - Split the dataset into **train** and **test** sets.

2. **Train a VAE**
   - Train a standard VAE using only the **clean (complete)** training set (i.e. no missing values).

3. **Simulate Missing Data in the Test Set**
   - Generate a **binary mask** for the test set where each entry has an 80% chance of being retained (True) and a 20% chance of being masked (False).
   - Replace all entries marked as missing (mask = False) with **zero**.

4. **Impute Missing Values**
   - Pass the **masked test set** through the VAE encoder.
   - Sample from the latent space and reconstruct the input using the decoder.
   - Use the decoder's output as the **imputed version** of the masked data.

5. **Evaluate Imputation Quality**
   - Compute the **Mean Squared Error (MSE)** and **R² score** over the **missing entries only** by comparing the imputed values against the original (unmasked) test set.

6. **Train a VAE with Random Masks (Imputation-Aware Training)**
   - Modify the training process:
     - At each iteration, apply a **random mask** to the training data (e.g. 20% of values masked).
     - Set the missing values to **0**.
     - Feed the masked data to the encoder.
     - When computing the loss (ELBO), evaluate **reconstruction over all entries (both observed and artificially masked)**.
     - KL divergence is computed as usual.

7. **Re-evaluate on the Test Set**
   - Using the **retrained VAE**, repeat Steps 3–5:
     - Apply a 20% missing mask to the test set.
     - Perform imputation using the retrained model.
     - Compute and report **MSE** and **R²** over the missing values.
